# 3. From crop detections to weed-candidate coordinates

**Goal:** understand and reproduce the image-processing stage, then optionally measure its quality using manual annotations.

The detector locates tobacco, not weeds. HSV thresholding finds green pixels; removing detected crop boxes leaves candidate weed regions. Contour filtering removes small regions, and contour moments provide candidate target centers. These are pixel coordinates, not calibrated robot or laser coordinates.

The default notebook displays saved illustrations without running inference. For a new run, install `requirements-experiments.txt`, supply images and trained tobacco weights, and set `RUN_PROCESSING = True`. No manual annotations are needed to generate predictions, but they are required to measure accuracy.


In [ ]:
from pathlib import Path
import sys
import json
import os
import tempfile

# Find the checkout whether Jupyter starts at its root or inside notebooks/.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'scripts/run_yolo_experiments.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Start Jupyter from this repository or its notebooks directory.')
sys.path.insert(0, str(ROOT / 'notebooks'))
from reproduction import run_script, new_run, frozen_assignments, write_local_splits, check_detector_results

# Caches are local runtime files, not historical experiment artifacts.
os.environ.setdefault('MPLCONFIGDIR', str(Path(tempfile.gettempdir()) / 'tobacco-notebook-mpl'))
os.environ.setdefault('YOLO_CONFIG_DIR', str(ROOT / 'experiments/.ultralytics'))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
RESULTS = ROOT / 'experiments/results'
print('Repository:', ROOT)


## Configure a new run

Use the `best.pt` from notebook 2, or obtain historical trained weights from the project owner. A generic pretrained YOLO checkpoint is not a trained single-class tobacco detector. New outputs use a fresh directory, independent of the detector run.

Optional annotation layout: `gt_masks/<image-stem>.png` for binary weed masks and `centers/<image-stem>.csv` for manually marked points with columns `x,y`. Coordinates use the original image size, origin at the upper left, x to the right and y downward.


In [ ]:
RUN_PROCESSING = False
IMAGES = ROOT / 'data/mask_validation/images'
WEIGHTS = ROOT / 'experiments/reproductions/detector-reproduction-01/yolo/yolov8x/weights/best.pt'
GT_MASKS = ROOT / 'data/mask_validation/gt_masks'
GT_CENTERS = ROOT / 'data/mask_validation/centers'
RUN_NAME = 'mask-reproduction-01'
DEVICE = '0'  # 'cpu' is suitable for an untimed small inference demonstration.
mask_run = None
processing_complete = False
config = json.loads((RESULTS / 'masks/mask_config.json').read_text())
print(json.dumps(config, indent=2))
print('Images available:', IMAGES.is_dir(), '| trained weights available:', WEIGHTS.is_file())


## See the historical pipeline

This is an existing qualitative panel, not a new model output or a ground-truth mask. The recorded processing used the tobacco image directory; that does not establish a separate weed validation set. When evaluating future masks, document which images were manually annotated and how they were held out from threshold tuning.


In [ ]:
display(Image(filename=str(ROOT / 'figures/publication/mask_pipeline_clean.jpg'), width=950))


## Inspect one image step by step

OpenCV reads BGR images. We convert to HSV for thresholding and RGB for display. OpenCV's 8-bit hue range is 0–179; the recorded bounds are H=35–85, S=40–255, V=40–255.

We call the existing functions in `scripts/generate_masks.py` so the explanation and batch implementation share the same algorithm. Detected tobacco rectangles use confidence 0.25, size 1024, and no padding. Their inverse is only a diagnostic: it includes soil and every other non-crop pixel.

Final contours must have area at least 50 pixels², with no maximum area. Area is OpenCV contour area, not a count of mask pixels. External contours are filled, which can fill internal holes. Their moment centroids are truncated to integer pixels and may not lie on vegetation for irregular shapes.


In [ ]:
if RUN_PROCESSING:
    if not IMAGES.is_dir() or not WEIGHTS.is_file():
        raise FileNotFoundError('Supply images and trained tobacco best.pt weights before processing.')
    import argparse
    import cv2
    import numpy as np
    sys.path.insert(0, str(ROOT / 'scripts'))
    import generate_masks as masks
    image_paths = masks.list_images(IMAGES)
    args = argparse.Namespace(device=DEVICE, conf=0.25, imgsz=1024,
        h_min=35, h_max=85, s_min=40, s_max=255, v_min=40, v_max=255,
        min_area=50.0, max_area=0.0, box_padding=0)
    detector = masks.load_detector(WEIGHTS)
    names = detector.names
    if len(names) != 1 or str(names[0]).lower() != 'tobacco':
        raise ValueError(f'Expected a single tobacco class; checkpoint names are {names}.')
    example = image_paths[0]
    bgr = cv2.imread(str(example))
    if bgr is None:
        raise ValueError(f'Cannot read {example}')
    vegetation = masks.hsv_mask(bgr, args)
    crops = masks.crop_mask(detector, example, vegetation.shape, args)
    inverse = cv2.bitwise_not(crops)
    residual = cv2.bitwise_and(vegetation, inverse)
    final, centers = masks.contour_filter(residual, args)
    fig, axes = plt.subplots(2, 3, figsize=(14, 8), layout='constrained')
    panels = [('Input RGB', cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)),
              ('HSV vegetation', vegetation), ('Detected crop regions', crops),
              ('Inverse crop mask (diagnostic)', inverse), ('Vegetation minus crops', residual),
              ('Filtered mask and centroids', final)]
    for ax, (title, array) in zip(axes.flat, panels):
        ax.imshow(array, cmap='gray', vmin=0, vmax=255)
        ax.set_title(title)
        ax.axis('off')
    if centers:
        x, y = zip(*centers)
        axes.flat[-1].scatter(x, y, color='red', s=15)
    plt.show()
    display(pd.DataFrame(centers, columns=['x', 'y']).head(10))
    # Release the preview model before the batch subprocess uses the GPU.
    del detector
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print('Preview skipped. Supply IMAGES and WEIGHTS, then enable RUN_PROCESSING.')


## Generate all four variants and center CSVs

The batch script writes `hsv_only`, `crop_exclusion_only`, `hsv_crop_exclusion`, and `final_contour_filtering`, plus `centers/*.csv` and the exact `mask_config.json`. A fresh run name prevents stale predictions from mixing with new outputs.


In [ ]:
if RUN_PROCESSING:
    mask_run = new_run(ROOT, RUN_NAME)
    run_script(ROOT, 'capture_environment.py', '--output', mask_run / 'environment')
    run_script(ROOT, 'generate_masks.py', '--images', IMAGES.resolve(), '--model', WEIGHTS.resolve(),
               '--output', mask_run / 'masks', '--device', DEVICE, '--conf', 0.25, '--imgsz', 1024,
               '--h-min', 35, '--h-max', 85, '--s-min', 40, '--s-max', 255,
               '--v-min', 40, '--v-max', 255, '--min-area', 50, '--max-area', 0, '--box-padding', 0)
    processing_complete = True
    print('Prediction outputs:', mask_run / 'masks')
else:
    print('Batch processing skipped; no predictions were generated.')


## Optional: evaluate against manual weed masks

Use 8-bit masks with **255 for weed and 0 for background**. The evaluator uses `value > 127`; masks encoded as 0/1 must be converted before evaluation. Shapes must match the original image, and filenames must share stems. Never place generated predictions in the ground-truth directory.

For each image, IoU = TP/(TP+FP+FN), Dice = 2TP/(2TP+FP+FN), precision = TP/(TP+FP), and recall = TP/(TP+FN). The script assigns 1 when a metric's denominator is zero, including empty/empty masks. Summary values are means across images, not pooled pixel totals.

Every annotated image must have a prediction. The evaluator otherwise marks missing predictions and averages only successful rows; this notebook rejects that incomplete summary. The inverse crop mask is included for diagnostic comparison, not as a meaningful vegetation detector.


In [ ]:
if processing_complete and GT_MASKS.is_dir() and any(GT_MASKS.glob('*.png')):
    summaries = []
    for variant in masks.VARIANTS:
        output = mask_run / f'mask_{variant}.csv'
        run_script(ROOT, 'evaluate_masks.py', '--ground-truth', GT_MASKS.resolve(),
                   '--predicted', mask_run / 'masks' / variant, '--output', output)
        scores = pd.read_csv(output)
        per_image = scores[scores['image'] != '__mean__']
        if per_image.empty or not per_image['status'].eq('ok').all():
            raise RuntimeError(f'{variant}: missing predictions; do not report a partial mean.')
        mean = scores.loc[scores['image'] == '__mean__'].iloc[0]
        summaries.append({'variant': variant, 'images': len(per_image),
            **{f'mean_{metric}': mean[metric] for metric in ['iou', 'precision', 'recall', 'dice']}})
    mask_summary = pd.DataFrame(summaries)
    mask_summary.to_csv(mask_run / 'mask_metrics.csv', index=False)
    display(mask_summary)
else:
    print('Mask quality not evaluated: run processing and provide manual gt_masks/*.png.')


## Optional: compare candidate centers with manual points

The current evaluator measures the distance from each ground-truth point to its nearest predicted point and reports hit rates within 10, 20, and 50 pixels. This is **not one-to-one matching**: one prediction can serve multiple ground-truth points, and extra predictions are not penalized. Therefore it does not measure false-target safety.

With ground-truth points but no predictions, distance is infinite and hit rates are zero. With no ground-truth points, hit rates are 1 by convention and mean distance is infinite. Inspect point counts alongside distances. Missing prediction files are rejected here rather than interpreted silently as empty predictions. Header-only `x,y` files represent valid empty predictions.


In [ ]:
if processing_complete and GT_CENTERS.is_dir() and any(GT_CENTERS.glob('*.csv')):
    missing = [p.name for p in GT_CENTERS.glob('*.csv')
               if not (mask_run / 'masks/centers' / p.name).is_file()]
    if missing:
        raise FileNotFoundError(f'Missing center predictions: {missing}')
    run_script(ROOT, 'evaluate_coordinates.py', '--ground-truth', GT_CENTERS.resolve(),
               '--predicted', mask_run / 'masks/centers', '--output', mask_run / 'coordinate_metrics.csv',
               '--not-available-output', mask_run / 'coordinate_metrics_not_available.md')
    display(pd.read_csv(mask_run / 'coordinate_metrics.csv'))
else:
    print('Coordinate accuracy not evaluated: run processing and provide manual centers/*.csv.')


## Interpret the outputs responsibly

Crop boxes can remove nearby weeds, and missed crops can become false weed candidates. Illumination affects fixed HSV thresholds. Contour centroids are geometric summaries rather than verified treatment points. Keep new prediction files, manual annotations, and historical evidence distinct.

The present article supports quantitative tobacco detection and qualitative mask/coordinate demonstrations. New manual annotations could support additional evaluation, but no notebook output alone validates physical laser treatment. Record dataset provenance, annotation policy, held-out image selection, settings, and hardware with any new results.
